# import libraries


In [1]:
import pandas as pd
import numpy as np
import os
from glob import glob



# set file paths


In [2]:
RAW_DATA_PATH = r"C:\Users\offic\Downloads\onedrive pl\OneDrive\Desktop\PRIXDATA\PRO\ecommerce_olist_data"
CLEANED_DATA_PATH = os.path.join(RAW_DATA_PATH, "cleaned")



# create folder if not exists


In [3]:
os.makedirs(CLEANED_DATA_PATH, exist_ok=True)



# required csv files


In [4]:
required_files = [
    "orders.csv",
    "order_items.csv",
    "products.csv",
    "order_payments.csv"
]



# get full paths


In [5]:
csv_files = [
    os.path.join(RAW_DATA_PATH, f)
    for f in required_files
]


In [6]:

print(f"Total datasets used: {len(csv_files)}")


Total datasets used: 4



# function to clean data


In [7]:
def clean_dataframe(df, file_name):

    original_shape = df.shape

    # clean column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    # remove duplicates
    df = df.drop_duplicates()

    # convert date columns
    for col in df.columns:
        if "date" in col or "timestamp" in col:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # handle missing values
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = df.select_dtypes(include=["object"]).columns

    protected_cols = [c for c in num_cols if "delivery" in c or "days" in c]
    safe_num_cols = list(set(num_cols) - set(protected_cols))

    df[safe_num_cols] = df[safe_num_cols].fillna(0)
    df[cat_cols] = df[cat_cols].fillna("UNKNOWN")

    # specific file rules
    if file_name == "orders.csv":
        df = df[df["order_status"] != "UNKNOWN"]

    if file_name == "order_items.csv" and "price" in df.columns:
        df["price"] = df["price"].clip(lower=0)

    if file_name == "order_payments.csv" and "payment_value" in df.columns:
        df["payment_value"] = df["payment_value"].clip(lower=0)

    # count missing data per row
    df["data_quality_missing_count"] = df.isnull().sum(axis=1)

    # print summary
    print(
        f"✔ {file_name} | "
        f"Rows: {original_shape[0]} → {df.shape[0]} | "
        f"Cols: {df.shape[1]}"
    )
    return df




# run cleaning for all files


In [8]:
for file in csv_files:
    file_name = os.path.basename(file)

    # read file
    df = pd.read_csv(file, low_memory=False)

    # clean file
    df_cleaned = clean_dataframe(df, file_name)

    # save cleaned file
    df_cleaned.to_csv(
        os.path.join(CLEANED_DATA_PATH, f"cleaned_{file_name}"),
        index=False
    )


✔ orders.csv | Rows: 99441 → 99441 | Cols: 9
✔ order_items.csv | Rows: 112650 → 112650 | Cols: 8
✔ products.csv | Rows: 32951 → 32951 | Cols: 10
✔ order_payments.csv | Rows: 103886 → 103886 | Cols: 6


In [9]:

print("\nREQUIRED DATASETS CLEANED SUCCESSFULLY")



REQUIRED DATASETS CLEANED SUCCESSFULLY
